In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(67)

def load_and_clean(filename):
    return np.loadtxt(filename, skiprows=2)

data_x = load_and_clean('x24x24.txt')
data_y = load_and_clean('y24x24.txt')
data_z = load_and_clean('z24x24.txt')
full_data = np.vstack([data_x, data_y, data_z])

X = full_data[:, :576]
y = full_data[:, 578]

In [2]:
import numpy as np

from skimage.feature import haar_like_feature, haar_like_feature_coord
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report
from skimage.transform import integral_image



IMG_SIZE = 24
N_HAAR_FEATURES = 1000  
RANDOM_STATE = 42


feature_types = [
    "type-2-x",
    "type-2-y",
    "type-3-x",
    "type-3-y",
    "type-4"
]

all_feature_coord, all_feature_type = haar_like_feature_coord(
    width=IMG_SIZE,
    height=IMG_SIZE,
    feature_type=feature_types
)

rng = np.random.default_rng(RANDOM_STATE)

selected_indices = rng.choice(
    len(all_feature_type),
    size=min(N_HAAR_FEATURES, len(all_feature_type)),
    replace=False
)

feature_coord = all_feature_coord[selected_indices]
feature_type = all_feature_type[selected_indices]



def extract_haar_features_cpu(X):
    X_images = X.reshape(-1, IMG_SIZE, IMG_SIZE)

    X_features = np.zeros(
        (X.shape[0], len(feature_type)),
        dtype=np.float32
    )

    for i, img in enumerate(X_images):
        img = img.astype(np.float32)

        #integral image
        ii = integral_image(img)

        X_features[i] = haar_like_feature(
            ii,
            0,
            0,
            IMG_SIZE,
            IMG_SIZE,
            feature_type=feature_type,
            feature_coord=feature_coord
        )

        if i % 500 == 0:
            print(f"Processed {i}/{len(X_images)} images")

    return X_features


# ==========================
# Feature extraction
# ==========================

print("Extracting Haar features on CPU...")

X_haar = extract_haar_features_cpu(X)

print("Original shape:", X.shape)
print("Haar shape:", X_haar.shape)



Extracting Haar features on CPU...
Processed 0/6835 images
Processed 500/6835 images
Processed 1000/6835 images
Processed 1500/6835 images
Processed 2000/6835 images
Processed 2500/6835 images
Processed 3000/6835 images
Processed 3500/6835 images
Processed 4000/6835 images
Processed 4500/6835 images
Processed 5000/6835 images
Processed 5500/6835 images
Processed 6000/6835 images
Processed 6500/6835 images
Original shape: (6835, 576)
Haar shape: (6835, 1000)


In [3]:

# ==========================
# Train / test split
# ==========================

X_train, X_test, y_train, y_test = train_test_split(
    X_haar,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)


In [4]:
base_estimator = DecisionTreeClassifier(
    max_depth=1
)

model = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=50,
    learning_rate=0.5
)

print("Training AdaBoost...")

model.fit(X_train, y_train)

y_pred = model.predict(X_test)


# ==========================
# Results
# ==========================

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Training AdaBoost...
Accuracy: 0.10021945866861741
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        31
         1.0       0.00      0.00      0.00        38
         2.0       0.00      0.00      0.00        26
         3.0       0.00      0.00      0.00        27
         4.0       0.00      0.00      0.00        20
         5.0       0.00      0.00      0.00        20
         6.0       0.13      0.17      0.15        40
         7.0       0.00      0.00      0.00        17
         8.0       0.00      0.00      0.00        37
         9.0       0.00      0.00      0.00        21
        10.0       0.00      0.00      0.00        25
        11.0       0.03      0.11      0.04        37
        12.0       0.00      0.00      0.00        31
        13.0       0.09      0.60      0.15        45
        14.0       0.00      0.00      0.00        23
        15.0       0.00      0.00      0.00        27
        16.0       0.00      0

c:\Users\marta\vs_projects\ai_project_photo_recognition_adaboost_ann\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\marta\vs_projects\ai_project_photo_recognition_adaboost_ann\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\marta\vs_projects\ai_project_photo_recognition_adaboost_ann\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_d